In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')))

from gptutils import *
from plan_demo import *

root_path = os.getcwd()

In [2]:
# First plan
prompt_directory = os.path.join(root_path, "../scripts/prompts")
prompts = load_prompts(prompt_directory)
task_instruction = "Get the keyboard to the transparent target."
topdown_img = os.path.join(root_path, "../planning_demo/sim_picture/initial_topdown.png")
first_plan = plan(task_instruction, topdown_img, prompts['firstplan'], prompts['action'])
subtasks = split_actions(first_plan)

```json
[
  {
    "action": "grasp",
    "parameters": {
      "object": "keyboard"
    }
  },
  {
    "action": "move to",
    "parameters": {
      "place": "transparent target"
    }
  },
  {
    "action": "release"
  }
]
```


In [ ]:
# If the grasp fails, get the error message
topdown_img = os.path.join(root_path, "../planning_demo/sim_picture/topdown_after_grasp.png")
front_img = os.path.join(root_path, "../planning_demo/sim_picture/front_after_grasp.png")
error_message = call_gpt_model(
    prompt1 = prompts['check_nopose'],
    task = subtasks[0],
    images= [topdown_img, front_img],
)
print(error_message)
error_message = error_message.strip().replace('```json', '').replace('```', '').strip()

```json
{
  "error_type": "object_too_low",
  "reason": "The keyboard is low on the table, leaving little clearance for the gripper to access from the sides or underneath without colliding with the surface."
}
```


In [5]:
# Replan
extra_env = get_env(subtasks[0], front_img, prompts['get_env_support'])
    
replan = call_gpt_model(
    prompt1 = prompts['grasp_replan_external'],
    prompt2 = extra_env,
    prompt3 = prompts['action'],
    task = subtasks,
    error_message = error_message,
    )
    
print(replan)

subtasks = split_actions(replan)

```json
[
  {
    "action": "push",
    "parameters": {
      "object": "keyboard",
      "place": "wooden wall"
    }
  },
  {
    "action": "rotate up",
    "parameters": {
      "object": "keyboard",
      "place": "wooden wall"
    }
  },
  {
    "action": "grasp",
    "parameters": {
      "object": "keyboard"
    }
  },
  {
    "action": "move to",
    "parameters": {
      "place": "transparent target"
    }
  },
  {
    "action": "release"
  }
]
```


In [ ]:
# If the push action of the replan fails, the object will be knocked off the table

topdown_img = os.path.join(root_path, "../planning_demo/sim_picture/topdown_after_push.png")
front_img = os.path.join(root_path, "../planning_demo/sim_picture/front_after_push.png")
plan = """
    [
        {
            "action": "push",
            "parameters": {
            "object": "keyboard",
            "place": "transparent target"
            }
        }
    ]
        """

subtasks = split_actions(plan)

# Get the error message
error_message = call_gpt_model(
    prompt1 = prompts['check_ik'],
    task = subtasks[0],
    images= [topdown_img, front_img],
)
print(error_message)
error_message = error_message.strip().replace('```json', '').replace('```', '').strip()

```json
{
  "error_type": "push_path_obstructed",
  "error_description": "An obstacle is on the path between the object's initial position and the target location, preventing it from being pushed directly to the target. The object either got knocked off the table or stopped in front of an obstacle."
}
```


In [8]:
# Replan
extra_env = get_env(subtasks[0], front_img, prompts['get_env_support'])
    
replan = call_gpt_model(
    prompt1 = prompts['grasp_replan_external'],
    prompt2 = extra_env,
    prompt3 = prompts['action'],
    task = subtasks,
    error_message = error_message,
    )
    
print(replan)

subtasks = split_actions(replan)

```json
[
  {
    "action": "push",
    "parameters": {
      "object": "keyboard",
      "place": "table edge"
    }
  },
  {
    "action": "grasp",
    "parameters": {
      "object": "keyboard"
    }
  },
  {
    "action": "move to",
    "parameters": {
      "place": "transparent target"
    }
  },
  {
    "action": "release"
  }
]
```
